# 04. Hipoteza 3: Benchmark modeli sentymentu

Ten notebook porównuje klasyczne modele ML oraz podejścia Hugging Face w zadaniu przewidywania `LINK_SENTIMENT`.
Najważniejsza metryka to F1 dla klasy negatywnej, ponieważ klasa `-1` jest mniejszościowa.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())


PROJECT_ROOT: /Users/weronikalewandowska/Documents/GitHub/projekt_reddit
DATA_PATH exists: False


## Modele porównywane w analizie rdzeniowej

- Dummy most frequent jako baseline nierównowagi klas.
- TF-IDF word 1-2 + Logistic Regression z `class_weight="balanced"`.
- TF-IDF word 1-2 + Linear SVC z wagami klas.
- Właściwości tekstu, LIWC i cechy sieciowe + Logistic Regression.
- Właściwości tekstu, LIWC i cechy sieciowe + Random Forest.
- TF-IDF + właściwości + Logistic Regression.
- Hugging Face direct baseline: istniejące `Content_Sentiment` jako predyktor negatywnego linku.
- Hugging Face derived features: `Content_Sentiment` i `Content_Score` w Logistic Regression.

Pełny kod benchmarku jest w `scripts/run_core_analyses.py`.


In [2]:
benchmark_path = OUTPUT_DIR / "classic_model_benchmark.csv"
if not benchmark_path.exists():
    import subprocess
    subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "run_core_analyses.py")], check=True)

benchmark = pd.read_csv(benchmark_path)
display(benchmark.sort_values(["negative_f1", "macro_f1"], ascending=False))


,model,accuracy,macro_f1,weighted_f1,negative_precision,negative_recall,negative_f1,tn_fp_fn_tp
0,TF-IDF word 1-2 + Logistic Regression balanced,0.848558,0.647033,0.872546,0.278011,0.601816,0.380328,"[8008, 1205, 307, 464]"
1,TF-IDF + Properties + Logistic Regression bala...,0.838642,0.640635,0.866188,0.266926,0.623865,0.373883,"[7892, 1321, 290, 481]"
2,TF-IDF word 1-2 + Linear SVC balanced,0.890224,0.646138,0.894640,0.323561,0.386511,0.352246,"[8590, 623, 473, 298]"
3,HF Content sentiment + Properties + Logistic R...,0.741587,0.566043,0.799419,0.184073,0.683528,0.290039,"[6877, 2336, 244, 527]"
4,Properties + Logistic Regression balanced,0.742688,0.564246,0.800028,0.181657,0.665370,0.285396,"[6902, 2311, 258, 513]"
5,HF Content_Sentiment direct baseline,0.823417,0.574074,0.849628,0.184879,0.377432,0.248188,"[7930, 1283, 480, 291]"
6,HF Content sentiment + Logistic Regression bal...,0.699219,0.522017,0.768100,0.143906,0.584955,0.230986,"[6530, 2683, 320, 451]"
7,Properties + Random Forest balanced,0.922977,0.522704,0.892288,0.514286,0.046693,0.085612,"[9179, 34, 735, 36]"
8,Dummy most frequent,0.922776,0.479919,0.885715,0.000000,0.000000,0.000000,"[9213, 0, 771, 0]"


In [ ]:
best = benchmark.sort_values(["negative_f1", "macro_f1"], ascending=False).iloc[0]
print("Najlepszy model według F1 klasy negatywnej:")
print(best[["model", "accuracy", "macro_f1", "negative_precision", "negative_recall", "negative_f1"]])
print()
print("Czy F1 klasy negatywnej przekracza 75%?", bool(best["negative_f1"] >= 0.75))
print("Hello World")

Najlepszy model według F1 klasy negatywnej:
model                 TF-IDF word 1-2 + Logistic Regression balanced
accuracy                                                    0.848558
macro_f1                                                    0.647033
negative_precision                                          0.278011
negative_recall                                             0.601816
negative_f1                                                 0.380328
Name: 0, dtype: object

Czy F1 klasy negatywnej przekracza 75%? False


## Opcjonalny benchmark embeddingów Hugging Face

Poniższa komórka uruchamia model `sentence-transformers/all-MiniLM-L6-v2` albo inny model embeddingowy z Hugging Face.
W lokalnym środowisku Windows może wymagać poprawnego PyTorch. W Colabie najprościej uruchomić ją z GPU.

Domyślnie `RUN_HF_EMBEDDINGS = False`, żeby przypadkowo nie pobierać dużych modeli i nie generować długo embeddingów.
Po ustawieniu na `True` wyniki zostaną dopisane do tabeli benchmarkowej.


In [4]:
RUN_HF_EMBEDDINGS = False
HF_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
HF_SAMPLE_SIZE = 12000

if RUN_HF_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
    from sklearn.model_selection import train_test_split
    from src.reddit_pbl.features import sentiment_to_binary_negative

    df = pd.read_csv(DATA_PATH)
    df["combined_text"] = df["Raw_Title"].fillna("").astype(str) + "\n" + df["Raw_Content"].fillna("").astype(str)
    y = sentiment_to_binary_negative(df["LINK_SENTIMENT"])

    if HF_SAMPLE_SIZE is not None and HF_SAMPLE_SIZE < len(df):
        sample = df.assign(y=y).sample(HF_SAMPLE_SIZE, random_state=42, stratify=y)
        y = sample["y"]
        texts = sample["combined_text"].tolist()
    else:
        texts = df["combined_text"].tolist()

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        texts, y, test_size=0.2, random_state=42, stratify=y
    )

    model = SentenceTransformer(HF_MODEL_NAME)
    X_train = model.encode(X_train_text, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    X_test = model.encode(X_test_text, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    hf_result = pd.DataFrame([{
        "model": f"HF embeddings {HF_MODEL_NAME} + Logistic Regression balanced",
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "weighted_f1": f1_score(y_test, pred, average="weighted"),
        "negative_precision": precision_score(y_test, pred, pos_label=1, zero_division=0),
        "negative_recall": recall_score(y_test, pred, pos_label=1, zero_division=0),
        "negative_f1": f1_score(y_test, pred, pos_label=1, zero_division=0),
        "tn_fp_fn_tp": confusion_matrix(y_test, pred, labels=[0, 1]).ravel().tolist(),
    }])
    display(pd.concat([benchmark, hf_result], ignore_index=True).sort_values(["negative_f1", "macro_f1"], ascending=False))
else:
    print("Pominięto embeddingowy benchmark HF. Ustaw RUN_HF_EMBEDDINGS = True, aby go uruchomić.")


Pominięto embeddingowy benchmark HF. Ustaw RUN_HF_EMBEDDINGS = True, aby go uruchomić.


## Wniosek

W lokalnie wykonanym benchmarku hipoteza H3 w wersji F1 > 75% nie została potwierdzona.
Najlepszy wynik dla klasy negatywnej osiągnął model TF-IDF + Logistic Regression balanced, z F1 klasy negatywnej około 0.38.

Random Forest ma wysoką accuracy, ale słabo rozpoznaje klasę negatywną. To pokazuje, że przy niezbalansowanych danych nie wolno wybierać modelu tylko po accuracy.
